Assignment I,
816004599
Big Data Analytics
PART 1 - Data Ingestion & Storage
Libraries Used - Requests & Pydantic

In [2]:
import polars as pl
import altair as alt
import duckdb
from pydantic import BaseModel, ValidationError
from pydantic.errors import PydanticUndefinedAnnotation
from datetime import datetime
#import logging
import requests 

try:
    tripdata = requests.get("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet")
except requests.exceptions.RequestException as err:
    logging.warning(err)

class TripTable(BaseModel):
    tpep_pickup_datetime: datetime
    tpep_dropoff_datetime: datetime
    passenger_count: int
    trip_distance: float
    fare_amount: float
    tip_amount: float
    total_amount: float
    payment_type: int

try:
    TripTable.model_rebuild()
except PydanticUndefinedAnnotation as exc_info:
    assert exc_info.code == 'undefined-annotation'

TripTable_df = pl.read_parquet(tripdata.content).select(["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount", "payment_type"]) #load entire dataset for transformations

print("TaxiZone Entries",TripTable_df.height)

try:
    TripTable.model_validate(TripTable_df.row(0,named=True),strict = True) #strict_mode in pydantic will raise and error log it for missing rows or incorrect datatypes. For parquet, It's standard that all rows contain same datatype and the first row of the dataframe should 
except ValidationError as exc:
    #triptable.model_validate(Tra) If needed, data type coercion can be handled by Pydantic(lax_mode) during validation or by DuckDB/Polars(fastest option) during DataFrame processing, although this wasn’t asked. 
    #logging.warning(exc)
    print(exc)

print(TripTable_df)
print("Validated")

TaxiZone Entries 2964624
shape: (2_964_624, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ tpep_pick ┆ tpep_drop ┆ PULocatio ┆ DOLocatio ┆ … ┆ fare_amou ┆ tip_amoun ┆ total_amo ┆ payment_ │
│ up_dateti ┆ off_datet ┆ nID       ┆ nID       ┆   ┆ nt        ┆ t         ┆ unt       ┆ type     │
│ me        ┆ ime       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ ---       ┆ ---       ┆ i32       ┆ i32       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ i64      │
│ datetime[ ┆ datetime[ ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ ns]       ┆ ns]       ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2024-01-0 ┆ 2024-01-0 ┆ 186       ┆ 79        ┆ … ┆ 17.7      ┆ 0.0       ┆ 22.7      ┆ 2        │
│ 1         ┆ 1         ┆           ┆      

In [3]:
try:    
    taxizone = requests.get("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
except requests.exceptions.RequestException as err:
    logging.warning(err)
   
class TaxiZone(BaseModel):
    LocationID: int
    Borough: str
    Zone: str
    service_zone: str
    
try:
    TaxiZone.model_rebuild()
except PydanticUndefinedAnnotation as exc_info:
    assert exc_info.code == 'undefined-annotation'

TaxiZone_df = pl.read_csv(taxizone.content)

for row in TaxiZone_df.to_dicts():
    try:
        TaxiZone.model_validate(row,strict = True)
        #strict_mode in pydantic will raise and error log it for missing rows or incorrect datatypes. For csv, We need to check each row.
    except ValidationError as exc:
        #taxiZone.model_validate(TaxiZone_df) If needed, data type coercion can be handled by Pydantic(lax_mode) during validation or by DuckDB/Polars(fastest option) during DataFrame processing, although this wasn’t asked. 
        #logging.warning(exc)
        print(exc)

print(TaxiZone_df)
print("Validated")

shape: (265, 4)
┌────────────┬───────────────┬─────────────────────────┬──────────────┐
│ LocationID ┆ Borough       ┆ Zone                    ┆ service_zone │
│ ---        ┆ ---           ┆ ---                     ┆ ---          │
│ i64        ┆ str           ┆ str                     ┆ str          │
╞════════════╪═══════════════╪═════════════════════════╪══════════════╡
│ 1          ┆ EWR           ┆ Newark Airport          ┆ EWR          │
│ 2          ┆ Queens        ┆ Jamaica Bay             ┆ Boro Zone    │
│ 3          ┆ Bronx         ┆ Allerton/Pelham Gardens ┆ Boro Zone    │
│ 4          ┆ Manhattan     ┆ Alphabet City           ┆ Yellow Zone  │
│ 5          ┆ Staten Island ┆ Arden Heights           ┆ Boro Zone    │
│ …          ┆ …             ┆ …                       ┆ …            │
│ 261        ┆ Manhattan     ┆ World Trade Center      ┆ Yellow Zone  │
│ 262        ┆ Manhattan     ┆ Yorkville East          ┆ Yellow Zone  │
│ 263        ┆ Manhattan     ┆ Yorkville West   

Part 2 - Data Transformation & Analysis 
Libraries used - Polars & DuckDB

In [11]:
before = TripTable_df.height
filteredTripTable = TripTable_df.drop_nulls(
    subset=["tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID", "fare_amount"]
)
TripTable_df.drop_nans(subset=["PULocationID","DOLocationID","fare_amount"])
print("Number of nulls removed:", before - filteredTripTable.height)

filteredTripTable = TripTable_df.filter(pl.col('trip_distance') > 0)
removed = TripTable_df.height - filteredTripTable.height
newheight = filteredTripTable.height
print("Total Rows removed for negative & zero distance :",removed)
filteredTripTable = filteredTripTable.filter(pl.col('tpep_pickup_datetime') < pl.col('tpep_dropoff_datetime'))
removed = newheight - filteredTripTable.height
newheight = filteredTripTable.height
print("Number of Rows removed for pickup times after dropoff:", removed)
filteredTripTable = filteredTripTable.filter((pl.col('fare_amount') > 0) & (pl.col('fare_amount') <= 500))
removed = newheight - filteredTripTable.height
print("Number of Rows removed for out of range fareamounts", removed)


Number of nulls removed: 0
Total Rows removed for negative & zero distance : 60371
Number of Rows removed for pickup times after dropoff: 112
Number of Rows removed for out of range fareamounts 34569


In [5]:
filteredTripTable = filteredTripTable.with_columns([
    ((pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime')).dt.total_seconds()/60).alias('trip_duration_mins'),
])

filteredTripTable = filteredTripTable.with_columns([
    (pl.col('trip_distance')/(pl.col('trip_duration_mins')/60)).alias('trip_speed_mph'),#didn't handle zeros/check if trip distance is miles or km
    (((pl.col('tpep_pickup_datetime')).dt.hour()).alias('pickup_hour')),
    ((pl.col('tpep_pickup_datetime')).dt.strftime("%A").alias('pickup_date_of_week')),  
])

print(filteredTripTable)

shape: (2_869_572, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ tpep_pick ┆ tpep_drop ┆ PULocatio ┆ DOLocatio ┆ … ┆ trip_dura ┆ trip_spee ┆ pickup_ho ┆ pickup_d │
│ up_dateti ┆ off_datet ┆ nID       ┆ nID       ┆   ┆ tion_mins ┆ d_mph     ┆ ur        ┆ ate_of_w │
│ me        ┆ ime       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ eek      │
│ ---       ┆ ---       ┆ i32       ┆ i32       ┆   ┆ f64       ┆ f64       ┆ i8        ┆ ---      │
│ datetime[ ┆ datetime[ ┆           ┆           ┆   ┆           ┆           ┆           ┆ str      │
│ ns]       ┆ ns]       ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2024-01-0 ┆ 2024-01-0 ┆ 186       ┆ 79        ┆ … ┆ 19.8      ┆ 5.212121  ┆ 0         ┆ Monday   │
│ 1         ┆ 1         ┆           ┆           ┆   ┆           ┆   

In [6]:
con = duckdb.connect()

con.register("filteredTripTable",filteredTripTable)
con.register("TaxiZone_df",TaxiZone_df)

result = con.sql("""SELECT PUlocationID AS "LocationID", SUM(PUlocationID) AS "NUMOFPICKUPS", TaxiZone_df.Zone AS "Pick-up Zone Name"
    FROM filteredTripTable
    JOIN TaxiZone_df
    ON filteredTripTable.PULocationID = TaxiZone_df.LocationID
    GROUP BY PULocationID, TaxiZone_df.Zone
    ORDER BY NUMOFPICKUPS DESC
    LIMIT 10;
    """)

print(result)

┌────────────┬──────────────┬──────────────────────────────┐
│ LocationID │ NUMOFPICKUPS │      Pick-up Zone Name       │
│   int32    │    int128    │           varchar            │
├────────────┼──────────────┼──────────────────────────────┤
│        237 │     33207966 │ Upper East Side South        │
│        236 │     31615032 │ Upper East Side North        │
│        230 │     23680340 │ Times Sq/Theatre District    │
│        161 │     22562701 │ Midtown Center               │
│        239 │     20665374 │ Upper West Side South        │
│        186 │     19000272 │ Penn Station/Madison Sq West │
│        132 │     18272364 │ JFK Airport                  │
│        234 │     17403282 │ Union Sq                     │
│        162 │     16903404 │ Midtown East                 │
│        249 │     16005720 │ West Village                 │
├────────────┴──────────────┴──────────────────────────────┤
│ 10 rows                                        3 columns │
└───────────────────────

In [7]:
result2 = con.sql("""SELECT pickup_hour, AVG(fare_amount) AS "AVERAGEFAREAMOUNT"
    FROM filteredTripTable
    GROUP BY pickup_hour
    ORDER BY AVERAGEFAREAMOUNT DESC
""")
print(result2)

┌─────────────┬────────────────────┐
│ pickup_hour │ AVERAGEFAREAMOUNT  │
│    int8     │       double       │
├─────────────┼────────────────────┤
│           5 │  27.50012006174608 │
│           4 │ 23.451589629435663 │
│           6 │  22.02714416197294 │
│          23 │ 20.246206988836196 │
│           0 │  19.68128814866009 │
│          16 │  19.45918333360364 │
│          14 │  19.27321030100783 │
│          15 │  19.11408553522016 │
│          22 │ 19.112202793914353 │
│           7 │ 18.753927078437673 │
│           · │          ·         │
│           · │          ·         │
│           · │          ·         │
│          20 │ 18.052625129388723 │
│          10 │  18.05072215123963 │
│           9 │  17.94713425800589 │
│           8 │   17.8265154786348 │
│          12 │ 17.798798629006026 │
│           1 │  17.73578070522983 │
│          11 │  17.63159584256259 │
│          19 │ 17.629133484352646 │
│          18 │  17.01523955666736 │
│           2 │ 16.629330091533202 │
├

In [8]:
result3 = con.sql("""SELECT payment_type,COUNT(payment_type) AS "NUMOFPAYMENTS", NUMOFPAYMENTS*100/(SELECT COUNT(*) FROM filteredTripTable) AS "PERCENTAGE",
    FROM filteredTripTable
    GROUP BY payment_type
""")

print(result3)

┌──────────────┬───────────────┬─────────────────────┐
│ payment_type │ NUMOFPAYMENTS │     PERCENTAGE      │
│    int64     │     int64     │       double        │
├──────────────┼───────────────┼─────────────────────┤
│            3 │         10561 │ 0.36803397858635367 │
│            0 │        115195 │   4.014361723629865 │
│            4 │         22756 │  0.7930102468242651 │
│            2 │        422713 │  14.730872757331058 │
│            1 │       2298347 │   80.09372129362846 │
└──────────────┴───────────────┴─────────────────────┘



In [9]:
result4 = con.sql("""SELECT pickup_date_of_week,AVG(tip_amount/fare_amount)*100 AS "RATIO TIP/FARE"
    FROM filteredTripTable
    WHERE payment_type == 1 
    GROUP BY pickup_date_of_week
""").show()

┌─────────────────────┬────────────────────┐
│ pickup_date_of_week │   RATIO TIP/FARE   │
│       varchar       │       double       │
├─────────────────────┼────────────────────┤
│ Saturday            │ 26.293994783251318 │
│ Sunday              │  25.10111835242046 │
│ Monday              │  25.51411585526936 │
│ Tuesday             │   25.7301240310411 │
│ Thursday            │  29.73445751931216 │
│ Friday              │ 25.595700912578433 │
│ Wednesday           │  25.70662471216203 │
└─────────────────────┴────────────────────┘



In [43]:
result5 = con.sql("""
WITH grouped_trips AS (
    SELECT 
        PULocationID, 
        DOLocationID, 
        COUNT(*) AS NumOfOccurrences
    FROM filteredTripTable
    GROUP BY PULocationID, DOLocationID
)
SELECT 
    g.PULocationID,
    g.DOLocationID,
    g.NumOfOccurrences,
    pu.Zone AS "Pick Up Zone",
    do_zone.Zone AS "Drop Off Zone"
FROM grouped_trips g
LEFT JOIN TaxiZone_df pu ON g.PULocationID = pu.LocationID
LEFT JOIN TaxiZone_df do_zone ON g.DOLocationID = do_zone.LocationID
ORDER BY g.NumOfOccurrences DESC
LIMIT 5;
""").show()

┌──────────────┬──────────────┬──────────────────┬───────────────────────┬───────────────────────┐
│ PULocationID │ DOLocationID │ NumOfOccurrences │     Pick Up Zone      │     Drop Off Zone     │
│    int32     │    int32     │      int64       │        varchar        │        varchar        │
├──────────────┼──────────────┼──────────────────┼───────────────────────┼───────────────────────┤
│          237 │          236 │            21641 │ Upper East Side South │ Upper East Side North │
│          236 │          237 │            19199 │ Upper East Side North │ Upper East Side South │
│          236 │          236 │            15193 │ Upper East Side North │ Upper East Side North │
│          237 │          237 │            14112 │ Upper East Side South │ Upper East Side South │
│          161 │          237 │            10139 │ Midtown Center        │ Upper East Side South │
└──────────────┴──────────────┴──────────────────┴───────────────────────┴───────────────────────┘



Part III - Visualization & Interpretations
Libraries Used - Altairs, Streamlit & DuckDB

In [71]:
df = result.pl()

# Strip any accidental whitespace from column names
df = df.rename({c: c.strip() for c in df.columns})

# Rename the zone column to something easy
df = df.rename({"Pick-up Zone Name": "pickup_zone"})

# Cast types so Altair can serialize (no Decimal)
df = df.with_columns(
    pl.col("NUMOFPICKUPS").cast(pl.Int64),
    pl.col("pickup_zone").cast(pl.Utf8),
)

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X("pickup_zone:N", sort="-y", title="Pick-up Zone"),
    y=alt.Y("NUMOFPICKUPS:Q", title="Number of Pickups"),
    tooltip=["LocationID:Q", "pickup_zone:N", "NUMOFPICKUPS:Q"]
).properties(
    title="Top 10 Pick-up Zones by Pickups",
    width = 450
)

chart

alt.Chart(...)

Interpretation: We can see that two zones (Upper East Side South & North) have almost 100 million more trips on average. 

In [45]:
df2 = result2.pl() 

df2 = df2.rename({c: c.strip() for c in df2.columns}).with_columns(
    pl.col("pickup_hour").cast(pl.Int64),
    pl.col("AVERAGEFAREAMOUNT").cast(pl.Float64),
).sort("pickup_hour")

chart = alt.Chart(df2).mark_line(point=True).encode(
    x=alt.X("pickup_hour:Q", title="Pickup Hour (0–23)"),
    y=alt.Y("AVERAGEFAREAMOUNT:Q", title="Average Fare Amount"),
    tooltip=["pickup_hour:Q", "AVERAGEFAREAMOUNT:Q"]
).properties(
    title="Average Fare Amount by Pickup Hour"
)

chart

alt.Chart(...)

Interpretation: The average fair amount varies inbetween 15- 20usd on an hourly basis. With the only anomaly being that there is a spike in average fare amount from 3am to 7am daily. This could be explained by a increased demand, longer trips, low taxi availability or heavier traffic conditions in the morning commute. Further analysis is required for find the root cause. I.E Like examing the average speed, trip durations, trip distances, etc.

In [70]:
result_dist = con.sql("""
SELECT
  CAST(FLOOR(trip_distance) AS INTEGER) AS dist_bin,
  COUNT(*) AS trips
FROM filteredTripTable
WHERE trip_distance > 0
  AND trip_distance <= 50
GROUP BY dist_bin
ORDER BY dist_bin
""").pl()

result_dist = result_dist.with_columns(
    pl.col("trips").cast(pl.Int64),
    pl.col("dist_bin").cast(pl.Int64),
)

chart = alt.Chart(result_dist).mark_bar().encode(
    x=alt.X("dist_bin:Q", title="Trip distance (miles, 1-mile bins)"),
    y=alt.Y("trips:Q", title="Number of trips"),
    tooltip=["dist_bin:Q", "trips:Q"]
).properties(
    title="Trip distance distribution (0–50 miles)",
    width = 450
)

chart

alt.Chart(...)

Interpretation: By trial and error, it was discovered that most trips are between 0 to 50 miles. Trip distance exhibits strong positive skew, with the vast majority of trips occurring between 0.5 and 5 miles, and a long tail extending toward longer distances. 

In [72]:
df3 = result3.pl().with_columns(
    pl.col("NUMOFPAYMENTS").cast(pl.Int64),
    pl.col("PERCENTAGE").cast(pl.Float64),
)

mapping = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided",
}

df3 = df3.with_columns(
    pl.col("payment_type")
      .cast(pl.Int64)
      .map_elements(lambda x: mapping.get(x, f"Other ({x})"), return_dtype=pl.Utf8)
      .alias("payment_name")
)

chart = alt.Chart(df3).mark_bar().encode(
    x=alt.X("payment_name:N", title="Payment Type"),
    y=alt.Y("NUMOFPAYMENTS:Q", title="Number of Payments"),
    tooltip=["payment_name:N", "NUMOFPAYMENTS:Q", alt.Tooltip("PERCENTAGE:Q", format=".5f")]
).properties(title="Payment Type Distribution",
            width = 500)

chart

alt.Chart(...)

Interpretation - We see that the prefered payment method is Credit Card payments at 80%, 14% is paid by cash and the others forms of payment are a minority, only accounting for 5 percent of payments. 

In [76]:
result = con.sql("""
SELECT
  pickup_date_of_week,
  pickup_hour,
  COUNT(*) AS num_trips
FROM filteredTripTable
GROUP BY pickup_date_of_week, pickup_hour
""").pl()

df = result.with_columns(
    pl.col("pickup_hour").cast(pl.Int64),
    pl.col("num_trips").cast(pl.Int64),
    pl.col("pickup_date_of_week").cast(pl.Utf8)
)

day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

heatmap = alt.Chart(df).mark_rect().encode(
    x=alt.X("pickup_hour:O", title="Hour of Day"),
    y=alt.Y("pickup_date_of_week:N", sort=day_order, title="Day of Week"),
    color=alt.Color("num_trips:Q", title="Trips"),
    tooltip=[
        alt.Tooltip("pickup_date_of_week:N", title="Day"),
        alt.Tooltip("pickup_hour:Q", title="Hour"),
        alt.Tooltip("num_trips:Q", title="Trips"),
    ],
).properties(
    width=1000,
    height=400,
    title="Trips by Day of Week and Hour"
)

heatmap

alt.Chart(...)

Interpretation: The peak demand for trips is within the after works hours of the early week. The lowest demand is within the late night hours of 1am to 5am during week days and 5am to 7am on weekends. Also, we can rule out an increased demand accounting for a spike in traffic fares as seen in the average fare/hour of day visualization. In fact it may be that lower demand results in lower cab availability which then leads to a spike in fares but more testing is required to say for certain.

AI Tool Usage - ChatGPT 5.2 - Used to assist with TypeCasting into altairs interpretable variables and assisted with query sql analysis(q). I needed to set up two instances of the Zone Data and join on both of them. Refining my english for my interpretations. 
Claude Opus 4.6 - Formating code, adding comments and sections, and also I used it to set up sessionstates to pass data between app.py and dashboard.py